# 11 — Heterogeneous Effects and Causal Machine Learning
**References:** Künzel et al. (2019, *PNAS*) · Athey & Imbens (2016, *PNAS*) · Wager & Athey (2018, *JASA*) · Chernozhukov et al. (2018, *Econometrics Journal*) · Stanford STATS 361 Lectures 4–7

## Narrative thread
```
CATE -> Meta-learners (S/T/X) -> Honest splitting & causal forests -> Double ML -> What ML buys (and doesn't)
```

## From the ATE to the CATE

The **conditional average treatment effect**
$$\tau(x) = E[Y(1) - Y(0) \mid X = x]$$
answers *who* benefits — the object behind personalized medicine, targeted discounts, and
uplift modeling. Estimating $\tau(x)$ is harder than prediction: **the label
$Y_i(1) - Y_i(0)$ is never observed for any unit**, so off-the-shelf supervised learning
does not directly apply. (Throughout, assume unconfoundedness + overlap, as in 05–06;
here treatment is randomized so we can isolate the estimation issues.)

## Meta-learners (Künzel et al. 2019)

Strategies that reduce CATE estimation to standard regression:

| Learner | Recipe | Weakness |
|---|---|---|
| **S** | One model $\hat\mu(x, w)$; $\hat\tau(x) = \hat\mu(x,1) - \hat\mu(x,0)$ | Regularization can shrink the effect to 0 |
| **T** | Separate $\hat\mu_1(x)$, $\hat\mu_0(x)$; take the difference | Differences two noisy fits; bad with unbalanced arms |
| **X** | Impute pseudo-effects $D_i$ from cross-arm predictions, regress, blend by $e(x)$ | More moving parts |
| **R / DR** | Orthogonalized residual-on-residual (Robinson 1988) | Needs good nuisance fits |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── A randomized dataset with strong effect heterogeneity ────────────────
from sklearn.ensemble import RandomForestRegressor

def make_hte_data(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, 4))
    tau = 2.0 * (X[:, 0] > 0) + 1.0 * X[:, 1]          # true CATE
    mu0 = 3 + 2*X[:, 0] + X[:, 2]**2                    # baseline surface
    w = rng.binomial(1, 0.5, n)                         # randomized
    y = mu0 + tau*w + rng.normal(0, 1, n)
    return X, w, y, tau

X, w, y, tau_true = make_hte_data(4000, 42)
Xte, _, _, tau_te = make_hte_data(2000, 7)

rf = dict(n_estimators=300, min_samples_leaf=20, random_state=0)

# S-learner
s_mod = RandomForestRegressor(**rf).fit(np.column_stack([X, w]), y)
tau_s = (s_mod.predict(np.column_stack([Xte, np.ones(len(Xte))]))
         - s_mod.predict(np.column_stack([Xte, np.zeros(len(Xte))])))

# T-learner
m1 = RandomForestRegressor(**rf).fit(X[w==1], y[w==1])
m0 = RandomForestRegressor(**rf).fit(X[w==0], y[w==0])
tau_t = m1.predict(Xte) - m0.predict(Xte)

# X-learner (e(x) = 0.5 known, randomized)
d1 = y[w==1] - m0.predict(X[w==1])          # imputed effects, treated arm
d0 = m1.predict(X[w==0]) - y[w==0]          # imputed effects, control arm
g1 = RandomForestRegressor(**rf).fit(X[w==1], d1)
g0 = RandomForestRegressor(**rf).fit(X[w==0], d0)
tau_x = 0.5*g1.predict(Xte) + 0.5*g0.predict(Xte)

for name, est in [('S-learner', tau_s), ('T-learner', tau_t), ('X-learner', tau_x)]:
    print(f'{name}: CATE RMSE = {np.sqrt(np.mean((est - tau_te)**2)):.3f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharex=True, sharey=True)
for ax, est, name in [(axes[0], tau_s, 'S'), (axes[1], tau_t, 'T'), (axes[2], tau_x, 'X')]:
    ax.scatter(tau_te, est, s=5, alpha=0.3)
    lims = [-2, 5]; ax.plot(lims, lims, 'r--', lw=1)
    ax.set_title(f'{name}-learner'); ax.set_xlabel('true CATE')
axes[0].set_ylabel('estimated CATE')
plt.tight_layout(); plt.show()

## Honesty and causal forests (Wager & Athey 2018)

A **causal tree** (Athey & Imbens 2016) splits to maximize heterogeneity in treatment
effects rather than to predict $Y$. The key innovation is **honesty**: use one subsample to
*choose the splits* and a disjoint subsample to *estimate the leaf effects*. Honest
estimation removes the adaptive overfitting bias and is what makes valid confidence
intervals possible; a **causal forest** averages many honest trees, with asymptotic
normality established by Wager & Athey (2018). (Production implementation: the R/C++
package `grf`; in Python, `econml`.)

## Double / debiased machine learning (Chernozhukov et al. 2018)

To use ML for the nuisance functions when estimating a low-dimensional target (e.g. the
ATE under unconfoundedness), two ingredients are essential:

1. **Neyman orthogonality** — use a moment equation whose bias is *second-order* in
   nuisance errors, e.g. the partialling-out (Robinson 1988) form:
   residualize $Y$ on $X$ and $W$ on $X$, then regress residual on residual.
2. **Cross-fitting** — estimate nuisances on one fold, evaluate on the other, and swap:
   kills the own-observation overfitting bias.

Plugging ML predictions in *naively* (no orthogonalization) inherits the ML's
regularization bias at first order — the estimate is systematically shrunk.


In [ ]:
# ── DML for the ATE in a confounded, nonlinear setting ───────────────────
from sklearn.model_selection import KFold

def make_confounded(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, 5))
    e = 1/(1 + np.exp(-1.5*np.sin(X[:, 0]) - 0.8*X[:, 1]))     # nonlinear propensity
    w = rng.binomial(1, e)
    y = 2.0*w + 3*np.sin(X[:, 0]) + X[:, 1]**2 + rng.normal(0, 1, n)  # true ATE = 2
    return X, w.astype(float), y

def dml_plm(X, w, y, cross_fit=True, seed=0):
    """Partially linear model: residual-on-residual with random forests."""
    yhat = np.zeros_like(y); what = np.zeros_like(w)
    if cross_fit:
        for tr, te in KFold(5, shuffle=True, random_state=seed).split(X):
            yhat[te] = RandomForestRegressor(200, min_samples_leaf=10, random_state=0)\
                         .fit(X[tr], y[tr]).predict(X[te])
            what[te] = RandomForestRegressor(200, min_samples_leaf=10, random_state=0)\
                         .fit(X[tr], w[tr]).predict(X[te])
    else:  # fit and predict on the SAME data (what not to do)
        yhat = RandomForestRegressor(200, min_samples_leaf=10, random_state=0).fit(X, y).predict(X)
        what = RandomForestRegressor(200, min_samples_leaf=10, random_state=0).fit(X, w).predict(X)
    ry, rw = y - yhat, w - what
    return (rw @ ry) / (rw @ rw)

ests_cf, ests_no = [], []
for r in range(30):
    X_, w_, y_ = make_confounded(2000, 100 + r)
    ests_cf.append(dml_plm(X_, w_, y_, cross_fit=True, seed=r))
    ests_no.append(dml_plm(X_, w_, y_, cross_fit=False))

naive_ml = []
for r in range(30):   # naive plug-in: y ~ RF(X, w), difference of predictions
    X_, w_, y_ = make_confounded(2000, 100 + r)
    m = RandomForestRegressor(200, min_samples_leaf=10, random_state=0)\
          .fit(np.column_stack([X_, w_]), y_)
    naive_ml.append(np.mean(m.predict(np.column_stack([X_, np.ones(len(X_))]))
                            - m.predict(np.column_stack([X_, np.zeros(len(X_))]))))

print(f'True ATE = 2.00')
print(f'Naive ML plug-in (S-learner ATE):    {np.mean(naive_ml):.3f}  (regularization bias)')
print(f'DML without cross-fitting:           {np.mean(ests_no):.3f}  (overfitting bias)')
print(f'DML with cross-fitting:              {np.mean(ests_cf):.3f}')

## What ML buys — and what it cannot

ML improves **estimation** (flexible nuisance functions, data-driven heterogeneity,
high-dimensional controls). It does **nothing for identification**: if unconfoundedness
fails, a causal forest estimates the wrong thing very flexibly. The order of operations
remains: design → identification (notebooks 01–10) → then, and only then, ML for the
nuisance parts.

## Key takeaways

| Concept | Statement |
|---|---|
| CATE $\tau(x)$ | Who benefits — never directly observed for any unit |
| S/T/X-learners | Reductions of CATE estimation to standard regression |
| Honesty | Split-selection and effect-estimation on disjoint samples |
| Causal forest | Honest trees + subsampling ⟹ CIs for $\tau(x)$ (Wager & Athey) |
| Neyman orthogonality | First-order insensitivity to nuisance estimation error |
| Cross-fitting | Removes own-observation bias from ML nuisances |
| Limits | ML sharpens estimation; identification still comes from design |
